# JuaKazi Gender Bias Engine — Multilingual Corrector Training

**Version:** v1 (Jun 2026)  
**Scope:** All 6 languages — SW, HA, ZU, KI, FR, EN  
**Model:** `castorini/afriteva_v2_base` — AfriTeVa v2, T5-style seq2seq pre-trained on African corpora  
**Task:** Seq2seq bias correction — biased sentence → neutral rewrite  
**Data:** `training_data_correction.csv` (~10K pairs across 6 languages)  
**Input prefix:** `correct bias {lang}: {biased sentence}`  
**Output:** corrected neutral sentence  
**Push to:** `juakazike/multilingual-bias-corrector-v1`  

## Why `castorini/afriteva_v2_base`?

AfriTeVa v2 is a T5-style encoder-decoder pre-trained on mC4 data filtered for African languages including Swahili, Hausa, Zulu, Kikuyu, French, and English. It outperforms mT5-base on African language generation tasks because it saw far more African text during pre-training. For seq2seq correction (biased → neutral rewriting), encoder-decoder is the right architecture — it reads the full biased sentence and generates the corrected version.

## Data summary

| Language | Pairs | Sources |
|----------|-------|---------|
| EN | 3,464 | WinoBias (3,162) + title expansion (240) + attitudinal templates (62) |
| ZU | 1,931 | zulu_retraining (instruction-tuning format, clean) |
| HA | 1,917 | juakazi_ha_correction_pairs_v1 (AI-drafted, structurally correct) |
| SW | 1,586 | juakazi_sw_correction_pairs_v1 (production quality) |
| KI | 867 | ground_truth_ki_v8 (passed QA) + lexicon-generated |
| FR | 636 | French Annotated gold rows (lexicon-generated) + GT |

## Hardware

Kaggle T4 x2 (recommended). ~2-3 hours for 5 epochs on ~10K pairs.  
Enable GPU: Notebook → Settings → Accelerator → GPU T4 x2

In [ ]:
# Cell A1: Install dependencies
import subprocess, sys

def install(packages):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + packages)

install([
    'transformers>=4.45.0',
    'datasets>=2.18.0',
    'accelerate>=0.34.0',
    'sentencepiece>=0.1.99',
    'evaluate>=0.4.0',
    'sacrebleu>=2.3.0',
    'rouge_score>=0.1.2',
    'huggingface_hub>=0.22.0',
    'matplotlib>=3.8.0',
    'seaborn>=0.13.0',
    'protobuf>=4.25.0',
])
print('Dependencies installed.')
print('RESTART RUNTIME now, then run Cell A2.')

In [ ]:
# Cell A2: Verify environment (run AFTER restart)
import torch, transformers, datasets
print(f'PyTorch:        {torch.__version__}')
print(f'Transformers:   {transformers.__version__}')
print(f'Datasets:       {datasets.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU count:      {torch.cuda.device_count()}')
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)} | VRAM: {torch.cuda.get_device_properties(i).total_memory/1e9:.1f} GB')
else:
    raise SystemExit('No GPU detected. Switch to T4 x2 in Notebook settings before running.')

In [ ]:
# Cell A3: Config + seeds
import random, numpy as np, torch

BASE_MODEL  = 'castorini/afriteva_v2_base'
HF_REPO     = 'juakazike/multilingual-bias-corrector-v1'
OUTPUT_DIR  = '/kaggle/working/afriteva-corrector'

SEED        = 42
MAX_LEN     = 128
TGT_LEN     = 128
BATCH_SIZE  = 8     # single GPU with gradient checkpointing
GRAD_ACCUM  = 8     # effective batch = 8*8 = 64
EPOCHS      = 5
LR          = 5e-5
WARMUP_STEPS = 200
WEIGHT_DECAY = 0.01
USE_FP16    = True

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f'Base model:     {BASE_MODEL}')
print(f'HF target:      {HF_REPO}')
print(f'Batch/device:   {BATCH_SIZE} (effective {BATCH_SIZE * GRAD_ACCUM})')
print(f'Epochs:         {EPOCHS}')
print(f'LR:             {LR}')
print(f'FP16:           {USE_FP16}')

In [ ]:
# Cell A4: Load correction pairs
# Upload training_data_correction.csv as Kaggle dataset:
#   kaggle.com -> Datasets -> New Dataset -> upload data/training_data_correction.csv
#   Name: juakazi-correction-data
# Add to this notebook: Notebook -> + Add Data -> search 'juakazi-correction-data'
# Path: /kaggle/input/juakazi-correction-data/training_data_correction.csv

import pandas as pd, numpy as np, os

DATA_PATH = '/kaggle/input/juakazi-correction-data/training_data_correction.csv'
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f'Dataset not found at {DATA_PATH}. Upload training_data_correction.csv as a Kaggle dataset.')

df = pd.read_csv(DATA_PATH)
df = df[df['input_text'].notna() & df['target_text'].notna()].copy()
df['input_text']  = df['input_text'].astype(str).str.strip()
df['target_text'] = df['target_text'].astype(str).str.strip()

# Drop pairs where correction is identical to input
before = len(df)
df = df[df['input_text'] != df['target_text']]
df = df[df['input_text'].str.len() > 5]
df = df[df['target_text'].str.len() > 2]
print(f'Dropped {before - len(df):,} identical/empty pairs. Remaining: {len(df):,}')

print(f'\nTotal pairs: {len(df):,}')
print(f'\nBy language:')
print(df['language'].value_counts().to_string())
print(f'\nBy source:')
print(df['source'].value_counts().to_string())

print('\nSample pairs (one per language):')
for lang in ['sw','ha','zu','ki','fr','en']:
    row = df[df['language']==lang].iloc[0]
    print(f'  [{lang.upper()}] IN:  {str(row["input_text"])[:80]}')
    print(f'  [{lang.upper()}] OUT: {str(row["target_text"])[:80]}')
    print()

In [ ]:
# Cell A5: Data exploration — length distributions and pair quality
import matplotlib.pyplot as plt, seaborn as sns

# Add model input prefix: 'correct bias {lang}: {text}'
df['model_input']  = df.apply(lambda r: f"correct bias {r['language']}: {r['input_text']}", axis=1)
df['model_target'] = df['target_text']

# Token length analysis (word-level proxy before loading tokenizer)
df['input_wlen']  = df['model_input'].str.split().apply(len)
df['target_wlen'] = df['model_target'].str.split().apply(len)
df['len_ratio']   = df['target_wlen'] / df['input_wlen'].clip(lower=1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Input length distribution
axes[0].hist(df['input_wlen'], bins=50, color='#2196F3', alpha=0.7, edgecolor='none')
axes[0].axvline(df['input_wlen'].quantile(0.95), color='red', linestyle='--', label='p95')
axes[0].set_xlabel('Input word count'); axes[0].set_ylabel('Freq')
axes[0].set_title('Input length distribution'); axes[0].legend()

# Target length distribution
axes[1].hist(df['target_wlen'], bins=50, color='#4CAF50', alpha=0.7, edgecolor='none')
axes[1].axvline(df['target_wlen'].quantile(0.95), color='red', linestyle='--', label='p95')
axes[1].set_xlabel('Target word count'); axes[1].set_ylabel('Freq')
axes[1].set_title('Target length distribution'); axes[1].legend()

# Pairs per language
lang_counts = df['language'].value_counts()
axes[2].bar(lang_counts.index, lang_counts.values, color=['#2196F3','#FF9800','#4CAF50','#9C27B0','#F44336','#009688'])
axes[2].set_xlabel('Language'); axes[2].set_ylabel('Pairs')
axes[2].set_title('Pairs per language')
for i, v in enumerate(lang_counts.values):
    axes[2].text(i, v+10, str(v), ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('/kaggle/working/data_overview.png', dpi=120)
plt.show()

print(f'Input length:  median={df["input_wlen"].median():.0f}  p95={df["input_wlen"].quantile(0.95):.0f}  max={df["input_wlen"].max()}')
print(f'Target length: median={df["target_wlen"].median():.0f}  p95={df["target_wlen"].quantile(0.95):.0f}  max={df["target_wlen"].max()}')
print(f'Length ratio (target/input):  mean={df["len_ratio"].mean():.2f}  std={df["len_ratio"].std():.2f}')
print(f'  (ratio < 0.5 or > 2.0 may indicate bad pairs)')
suspicious = df[(df['len_ratio'] < 0.3) | (df['len_ratio'] > 3.0)]
print(f'  Suspicious pairs (ratio outside 0.3-3.0): {len(suspicious)}')
if len(suspicious) > 0:
    print(suspicious[['language','input_text','target_text','len_ratio']].head(5).to_string())

In [ ]:
# Cell A6: Train/val split — stratified by language
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df, test_size=0.10, random_state=SEED, stratify=df['language']
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print(f'Train: {len(train_df):,} pairs | Val: {len(val_df):,} pairs')
print(f'\nTrain by language:')
for lang in sorted(train_df['language'].unique()):
    t = train_df[train_df['language']==lang]
    v = val_df[val_df['language']==lang]
    print(f'  {lang}: train={len(t):,}  val={len(v):,}')

In [ ]:
# Cell A7: Load tokenizer and verify token lengths
from transformers import AutoTokenizer
import matplotlib.pyplot as plt

print(f'Loading tokenizer: {BASE_MODEL}')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
print(f'Vocab size: {tokenizer.vocab_size:,}')

# Sample 2K pairs to estimate token lengths
sample = train_df.sample(min(2000, len(train_df)), random_state=SEED)
inp_lengths = [len(tokenizer.encode(t, truncation=False)) for t in sample['model_input']]
tgt_lengths = [len(tokenizer.encode(t, truncation=False)) for t in sample['model_target']]

import numpy as np
for name, lens in [('Input', inp_lengths), ('Target', tgt_lengths)]:
    print(f'{name} token lengths: p50={np.percentile(lens,50):.0f}  p90={np.percentile(lens,90):.0f}  p95={np.percentile(lens,95):.0f}  p99={np.percentile(lens,99):.0f}  max={max(lens)}')
    pct_trunc = sum(1 for l in lens if l > MAX_LEN) / len(lens)
    print(f'  % truncated at MAX_LEN={MAX_LEN}: {pct_trunc*100:.1f}%')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(inp_lengths, bins=50, color='#2196F3', alpha=0.7)
axes[0].axvline(MAX_LEN, color='red', linestyle='--', label=f'MAX_LEN={MAX_LEN}')
axes[0].set_title('Input token lengths'); axes[0].set_xlabel('Tokens'); axes[0].legend()
axes[1].hist(tgt_lengths, bins=50, color='#4CAF50', alpha=0.7)
axes[1].axvline(TGT_LEN, color='red', linestyle='--', label=f'TGT_LEN={TGT_LEN}')
axes[1].set_title('Target token lengths'); axes[1].set_xlabel('Tokens'); axes[1].legend()
plt.tight_layout()
plt.savefig('/kaggle/working/token_lengths.png', dpi=120)
plt.show()

In [ ]:
# Cell A8: Load AfriTeVa model — single GPU + gradient checkpointing
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'   # single GPU — avoids DataParallel OOM on backward pass

from transformers import AutoModelForSeq2SeqLM
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

print(f'Loading model: {BASE_MODEL}')
model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)
model.gradient_checkpointing_enable()   # recompute activations on backward — halves VRAM
model = model.to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params:     {total_params/1e6:.1f}M')
print(f'Trainable params: {trainable_params/1e6:.1f}M')
print(f'Encoder layers:   {model.config.num_layers}')
print(f'Decoder layers:   {model.config.num_decoder_layers}')
print(f'Model dim:        {model.config.d_model}')
print(f'VRAM after load:  {torch.cuda.memory_allocated()/1e9:.2f} GB')

In [ ]:
# Cell A9: Tokenize training and validation datasets
from datasets import Dataset

def tokenize_batch(batch):
    model_inputs = tokenizer(
        batch['model_input'],
        max_length=MAX_LEN,
        truncation=True,
    )
    labels = tokenizer(
        text_target=batch['model_target'],
        max_length=TGT_LEN,
        truncation=True,
    )
    # Let DataCollatorForSeq2Seq handle padding and -100 masking
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

print('Tokenizing train set...')
train_ds = Dataset.from_pandas(train_df[['model_input','model_target']].reset_index(drop=True))
train_tok = train_ds.map(tokenize_batch, batched=True, batch_size=256,
                          remove_columns=['model_input','model_target'])

print('Tokenizing val set...')
val_ds = Dataset.from_pandas(val_df[['model_input','model_target']].reset_index(drop=True))
val_tok = val_ds.map(tokenize_batch, batched=True, batch_size=256,
                      remove_columns=['model_input','model_target'])

print(f'Train tokenized: {len(train_tok):,} examples')
print(f'Val tokenized:   {len(val_tok):,} examples')

# Sanity check — labels must not be empty
sample_labels = train_tok[0]['labels']
print(f'Sample label token ids (first 10): {sample_labels[:10]}')
if len(sample_labels) == 0:
    raise ValueError('Labels are empty — tokenization failed')
print('Labels OK')

In [ ]:
# Cell A10: Evaluation metrics — BLEU + ROUGE
import evaluate, numpy as np

sacrebleu = evaluate.load('sacrebleu')
rouge     = evaluate.load('rouge')

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    # Replace -100 (ignored padding) with pad token id before decoding
    preds  = np.where(preds  != -100, preds,  tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds  = tokenizer.batch_decode(preds,  skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds  = [p.strip() for p in decoded_preds]
    decoded_labels = [[l.strip()] for l in decoded_labels]  # sacrebleu expects list of refs

    bleu_result   = sacrebleu.compute(predictions=decoded_preds, references=decoded_labels)
    rouge_result  = rouge.compute(predictions=decoded_preds,
                                   references=[l[0] for l in decoded_labels])

    return {
        'bleu':    round(bleu_result['score'], 2),
        'rouge1':  round(rouge_result['rouge1'], 4),
        'rouge2':  round(rouge_result['rouge2'], 4),
        'rougeL':  round(rouge_result['rougeL'], 4),
    }

print('Metrics defined: BLEU + ROUGE-1/2/L')
print('Target BLEU after training: >= 25.0 (good for bias correction)')
print('BLEU interpretation: 25-40 = good quality rewriting, 40+ = excellent')

In [ ]:
# Cell A11: TrainingArguments + Seq2SeqTrainer
from transformers import (
    Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
)
import os, torch

torch.cuda.empty_cache()
os.makedirs(OUTPUT_DIR, exist_ok=True)

training_args = Seq2SeqTrainingArguments(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    learning_rate               = LR,
    warmup_steps                = WARMUP_STEPS,
    weight_decay                = WEIGHT_DECAY,
    lr_scheduler_type           = 'cosine',
    predict_with_generate       = True,
    generation_max_length       = TGT_LEN,
    generation_num_beams        = 4,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'bleu',
    greater_is_better           = True,
    fp16                        = torch.cuda.is_available() and USE_FP16,
    gradient_accumulation_steps = GRAD_ACCUM,
    dataloader_num_workers      = 2,
    logging_steps               = 50,
    save_total_limit            = 1,
    save_only_model             = True,
    report_to                   = 'none',
    seed                        = SEED,
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer, model=model, padding=True, pad_to_multiple_of=8
)

trainer = Seq2SeqTrainer(
    model            = model,
    args             = training_args,
    train_dataset    = train_tok,
    eval_dataset     = val_tok,
    processing_class = tokenizer,
    data_collator    = data_collator,
    compute_metrics  = compute_metrics,
)

steps_per_epoch = len(train_tok) // (BATCH_SIZE * GRAD_ACCUM)
total_steps     = steps_per_epoch * EPOCHS
print(f'Steps/epoch:    {steps_per_epoch:,}')
print(f'Total steps:    {total_steps:,}')
print(f'Effective batch:{BATCH_SIZE * GRAD_ACCUM}')
print(f'Est. time:      ~{total_steps * 5 / 60:.0f} min on 1x T4')
print('Trainer ready.')

In [ ]:
# Cell A12: TRAIN
# ~2-3 hours on T4 x2. Do not close the browser tab.
print('Starting training...')
print(f'Model:   {BASE_MODEL}')
print(f'Pairs:   {len(train_tok):,} train | {len(val_tok):,} val')
print(f'Epochs:  {EPOCHS}')
print('-' * 60)

result = trainer.train()

print('-' * 60)
print(f'Training complete.')
print(f'  Total steps:      {result.global_step:,}')
print(f'  Training loss:    {result.training_loss:.4f}')
print(f'  Training time:    {result.metrics["train_runtime"]/60:.1f} min')
print(f'  Samples/sec:      {result.metrics["train_samples_per_second"]:.1f}')

In [ ]:
# Cell A13: Training curves — loss + BLEU per epoch
import matplotlib.pyplot as plt

history = trainer.state.log_history

epoch_data  = []
bleu_data   = []
rouge1_data = []
rougeL_data = []
train_loss  = []

for entry in history:
    if 'eval_bleu' in entry:
        epoch_data.append(entry.get('epoch', 0))
        bleu_data.append(entry['eval_bleu'])
        rouge1_data.append(entry.get('eval_rouge1', 0))
        rougeL_data.append(entry.get('eval_rougeL', 0))
    if 'loss' in entry and 'eval_loss' not in entry:
        train_loss.append(entry['loss'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(range(len(train_loss)), train_loss, color='#F44336', linewidth=1.5)
axes[0].set_xlabel('Step'); axes[0].set_ylabel('Loss')
axes[0].set_title('Training loss'); axes[0].grid(alpha=0.3)

axes[1].plot(epoch_data, bleu_data,   label='BLEU',    marker='o', color='#2196F3')
axes[1].plot(epoch_data, rouge1_data, label='ROUGE-1', marker='s', color='#4CAF50', linestyle='--')
axes[1].plot(epoch_data, rougeL_data, label='ROUGE-L', marker='^', color='#FF9800', linestyle='--')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Score')
axes[1].set_title('Validation metrics per epoch')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/training_curves.png', dpi=120)
plt.show()

final_metrics = trainer.evaluate()
print('\n=== Final validation metrics ===')
for k, v in final_metrics.items():
    print(f'  {k}: {v}')

In [ ]:
# Cell A14: Per-language evaluation
# Measures how well the model corrects each language independently.
from datasets import Dataset
import numpy as np

def evaluate_language(lang_code):
    lang_val = val_df[val_df['language'] == lang_code].reset_index(drop=True)
    if len(lang_val) == 0:
        return None

    lang_ds = Dataset.from_pandas(lang_val[['model_input','model_target']])
    lang_tok = lang_ds.map(tokenize_batch, batched=True, batch_size=64,
                            remove_columns=['model_input','model_target'])
    lang_tok.set_format('torch')

    preds_output = trainer.predict(lang_tok)
    preds  = preds_output.predictions
    labels = preds_output.label_ids

    preds  = np.where(preds  != -100, preds,  tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds  = [p.strip() for p in tokenizer.batch_decode(preds,  skip_special_tokens=True)]
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    bleu = sacrebleu.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])
    r    = rouge.compute(predictions=decoded_preds, references=decoded_labels)

    return {
        'lang': lang_code, 'n': len(lang_val),
        'bleu': round(bleu['score'], 2),
        'rouge1': round(r['rouge1'], 4),
        'rougeL': round(r['rougeL'], 4),
        'examples': list(zip(lang_val['input_text'].tolist()[:3],
                             lang_val['target_text'].tolist()[:3],
                             decoded_preds[:3]))
    }

model.eval()
lang_results = {}
print('Per-language evaluation:')
print(f'{"Lang":6} {"N":6} {"BLEU":8} {"ROUGE-1":8} {"ROUGE-L":8}')
print('-' * 45)
for lang in ['sw','ha','zu','ki','fr','en']:
    r = evaluate_language(lang)
    if r:
        lang_results[lang] = r
        print(f'{lang:6} {r["n"]:6} {r["bleu"]:8.2f} {r["rouge1"]:8.4f} {r["rougeL"]:8.4f}')

print('\nExample predictions (input | expected | predicted):')
for lang, r in lang_results.items():
    print(f'\n[{lang.upper()}]')
    for inp, exp, pred in r['examples']:
        print(f'  IN:  {inp[:80]}')
        print(f'  EXP: {exp[:80]}')
        print(f'  GOT: {pred[:80]}')
        print()

In [ ]:
# Cell A15: Manual inference test on hard cases
import torch

def correct(text, lang, num_beams=4):
    prompt = f'correct bias {lang}: {text}'
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=MAX_LEN)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=TGT_LEN,
            num_beams=num_beams,
            early_stopping=True,
            no_repeat_ngram_size=3,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)

test_cases = [
    ('sw', 'Daktari wa kiume alifika hospitalini asubuhi.'),
    ('sw', 'Mwalimu wa kike alitoa somo zuri leo.'),
    ('ha', 'Likitan namiji ne kawai zai iya jagorantar asibiti.'),
    ('ha', 'Matan gida suna girki kawai ba sa iya jagoranci.'),
    ('zu', 'Udokotela wesilisa weza esibhedlela ekuseni.'),
    ('zu', 'Abesifazane abakwazi ukusebenza kwi-IT.'),
    ('ki', 'Mundũ wa mũrũme nĩwe ũngĩ gũtwara mũciĩ.'),
    ('fr', 'Le président a dirigé la réunion comme un vrai homme.'),
    ('fr', 'Les femmes sont trop émotives pour gérer une équipe.'),
    ('en', 'The chairman will lead the board meeting.'),
    ('en', 'Women are too emotional for leadership roles.'),
    ('en', 'Every doctor should update his records.'),
]

model.eval()
print('=== Manual inference test ===')
all_pass = True
for lang, sentence in test_cases:
    result = correct(sentence, lang)
    changed = result.strip() != sentence.strip()
    status = 'CHANGED' if changed else 'UNCHANGED'
    print(f'[{lang.upper()}] {status}')
    print(f'  IN:  {sentence}')
    print(f'  OUT: {result}')
    print()
    if not changed:
        all_pass = False

if all_pass:
    print('All test sentences were corrected.')
else:
    print('WARNING: some sentences were not corrected. Check BLEU and consider more epochs.')

In [ ]:
# Cell A16: Save model + push to HuggingFace
from huggingface_hub import HfApi, login
import json, os
from datetime import datetime

# Get token from Kaggle secrets: Add-ons -> Secrets -> HF_TOKEN
HF_TOKEN = os.environ.get('HF_TOKEN', '')
if not HF_TOKEN:
    HF_TOKEN = input('Paste HuggingFace write token: ').strip()

login(token=HF_TOKEN)

SAVE_DIR = '/kaggle/working/afriteva-corrector-final'
os.makedirs(SAVE_DIR, exist_ok=True)

model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

# Save metadata JSON
metadata = {
    'model_id':       HF_REPO,
    'base_model':     BASE_MODEL,
    'version':        'v1',
    'task':           'gender-bias-correction',
    'languages':      ['sw','ha','zu','ki','fr','en'],
    'input_format':   'correct bias {lang}: {biased sentence}',
    'trained_on':     datetime.utcnow().strftime('%Y-%m-%d'),
    'training_pairs': len(train_tok),
    'val_pairs':      len(val_tok),
    'max_input_len':  MAX_LEN,
    'max_target_len': TGT_LEN,
    'pairs_by_lang':  {lang: int((df['language']==lang).sum()) for lang in ['sw','ha','zu','ki','fr','en']},
    'final_metrics':  final_metrics,
    'per_lang_bleu':  {lang: r['bleu'] for lang, r in lang_results.items()},
}
with open(f'{SAVE_DIR}/juakazi_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print('Metadata saved:')
print(json.dumps(metadata, indent=2))

# Push to Hub
api = HfApi()
api.create_repo(HF_REPO, exist_ok=True, private=False)
api.upload_folder(
    folder_path=SAVE_DIR,
    repo_id=HF_REPO,
    repo_type='model',
    commit_message=f'Train multilingual-bias-corrector-v1 ({datetime.utcnow().strftime("%Y-%m-%d")})',
)
print(f'\nModel pushed to: https://huggingface.co/{HF_REPO}')

In [ ]:
# Cell A17: Write model card to HuggingFace
from huggingface_hub import HfApi

bleu_sw = lang_results.get('sw', {}).get('bleu', 0)
bleu_ha = lang_results.get('ha', {}).get('bleu', 0)
bleu_zu = lang_results.get('zu', {}).get('bleu', 0)
bleu_ki = lang_results.get('ki', {}).get('bleu', 0)
bleu_fr = lang_results.get('fr', {}).get('bleu', 0)
bleu_en = lang_results.get('en', {}).get('bleu', 0)

card = f'''---
language:
- sw
- ha
- zu
- ki
- fr
- en
tags:
- gender-bias
- seq2seq
- text-generation
- african-nlp
- bias-correction
license: apache-2.0
---

# JuaKazi Multilingual Bias Corrector v1

Seq2seq gender bias correction model covering 6 languages.
Fine-tuned from `castorini/afriteva_v2_base` on ~10K correction pairs.

## Usage

Input format: `correct bias {{lang}}: {{biased sentence}}`

Where `lang` is one of: `sw`, `ha`, `zu`, `ki`, `fr`, `en`

```python
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("juakazike/multilingual-bias-corrector-v1")
model = AutoModelForSeq2SeqLM.from_pretrained("juakazike/multilingual-bias-corrector-v1")

def correct(text, lang):
    prompt = f"correct bias {{lang}}: {{text}}"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=128)
    out = model.generate(**inputs, max_new_tokens=128, num_beams=4)
    return tokenizer.decode(out[0], skip_special_tokens=True)

correct("The chairman will lead the board meeting.", "en")
# -> "The chair will lead the board meeting."
```

## Validation BLEU (val set, 10% held out per language)

| Language | Pairs | BLEU |
|----------|-------|------|
| Swahili (sw) | 1,586 | {bleu_sw:.1f} |
| Hausa (ha) | 1,917 | {bleu_ha:.1f} |
| Zulu (zu) | 1,931 | {bleu_zu:.1f} |
| Gikuyu (ki) | 867 | {bleu_ki:.1f} |
| French (fr) | 636 | {bleu_fr:.1f} |
| English (en) | 3,464 | {bleu_en:.1f} |
'''

api = HfApi()
api.upload_file(
    path_or_fileobj=card.encode(),
    path_in_repo='README.md',
    repo_id=HF_REPO,
    repo_type='model',
    commit_message='Add model card',
)
print('Model card written.')
print(f'\nhttps://huggingface.co/{HF_REPO}')

In [ ]:
# Cell A18: Log metrics — download and merge into eval/metrics.json
from datetime import datetime
import json

import json

metrics_update = {
    'corrector': {
        'model':          HF_REPO,
        'base':           BASE_MODEL,
        'overall_bleu':   final_metrics.get('eval_bleu', 0),
        'overall_rouge1': final_metrics.get('eval_rouge1', 0),
        'overall_rougeL': final_metrics.get('eval_rougeL', 0),
        'per_lang_bleu':  {lang: r['bleu'] for lang, r in lang_results.items()},
        'train_pairs':    len(train_tok),
        'val_pairs':      len(val_tok),
        'epochs':         EPOCHS,
        'languages':      ['sw','ha','zu','ki','fr','en'],
        'trained':        datetime.utcnow().strftime('%Y-%m-%d'),
    }
}

with open('/kaggle/working/corrector_metrics.json', 'w') as f:
    json.dump(metrics_update, f, indent=2)

print('Saved: /kaggle/working/corrector_metrics.json')
print('Download this file and merge into eval/metrics.json in the repo.')
print()
print(json.dumps(metrics_update, indent=2))
print()
print('=== TRAINING COMPLETE ===')
print(f'Corrector: https://huggingface.co/{HF_REPO}')
print()
print('Next steps:')
print('  1. Download corrector_metrics.json, merge into eval/metrics.json')
print('  2. Set JUAKAZI_CORRECTOR_MODEL env var on App Runner to:', HF_REPO)
print('  3. Re-run: python3 run_evaluation.py')
print('  4. Test live: POST /analyse with a biased sentence per language')